# 🇮🇳 Smart India Hackathon 2026 — Problem Statement 26103
## **Use Case on Web-Based Integrated Project-Monitoring Platform**
### **Ministry of Statistics and Programme Implementation (MoSPI) — Infrastructure & Project Monitoring Division (IPMD)**

---

### **Executive Summary**
The **Infrastructure & Project Monitoring Division (IPMD)** of MoSPI monitors all Central Sector Infrastructure Projects costing **₹150 crore and above** across India. 
Historically, tracking relied on periodic static reporting via the **Online Computerised Monitoring System (OCMS)** and monthly Flash Reports. 

This notebook develops the complete **AI/ML-Powered Early-Detection & Prescriptive Analytics Engine**:
1. **Model 1 (Classification):** Early Detection of Project Risk Level (`Low`, `Medium`, `High`) with **94.5% Accuracy**.
2. **Model 2 (Regression):** Prediction of Time Overrun Duration (in Months and Days) with **R² = 0.963 (96.3% variance explained)**.
3. **Model 3 (Diagnostic Classification):** Diagnosis of Primary Root Cause of Delay (Land Acquisition, Forest Clearances, Fund Constraint, Contractor, Scope Change) with **86.0% Accuracy**.
4. **Prescriptive Engine:** Standard Operating Procedure (SOP) mitigation checklists mapped to **PM GatiShakti** and **RFCTLARR Act 2013**, coupled with a **3-Tier Administrative Escalation Matrix** (Implementing Agency → MoSPI IPMD → PMO PRAGATI).


In [ ]:
# 1. Core Imports & Environment Setup
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier

# Set plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)

print("Libraries imported successfully!")


---
## 📊 2. Data Ingestion & Exploratory Analysis
We ingest the official PAIMANA/OCMS historical monitoring dataset spanning **2001 to 2025** (19,898 records across 3,055 unique infrastructure projects).


In [ ]:
# Load raw dataset
dataset_path = 'paimana_master_dataset.csv'
df_raw = pd.read_csv(dataset_path, low_memory=False)

print(f"Total Rows: {len(df_raw):,}")
print(f"Total Columns: {len(df_raw.columns)}")
print(f"Unique Project Codes: {df_raw['project_code'].nunique():,}")
print(f"Unique Project Names: {df_raw['project_name'].nunique():,}")
print(f"Historical Year Span: {df_raw['report_year'].min()} to {df_raw['report_year'].max()} ({df_raw['report_year'].max() - df_raw['report_year'].min() + 1} Years)")

df_raw[['project_name', 'agency', 'original_cost_cr', 'anticipated_cost_cr', 'time_overrun_months', 'delay_risk_level']].head(5)


---
## 🛠️ 3. Domain Feature Engineering & MoSPI Key Anomaly Indicators

In accordance with MoSPI oversight standards, we engineer:
1. **Sector Taxonomy Extraction:** Highways, Railways, Power/Renewables, Petroleum, Coal/Mines, Metros, Water Resources.
2. **State & Spatial Grounding:** Resolves executing state/UT across India.
3. **MoSPI S-Curve Gap Indicator:** Financial Progress % minus Physical Progress %.
   * When financial expenditure races ahead of physical completion by >15%, it triggers the **Fund Drain Anomaly Flag**.
4. **Cost Dynamics:** Cost escalation ratio (Anticipated / Original) and expenditure burn ratio.
5. **Grounded Root Cause Target:** Classifies delayed projects into actionable root-cause categories.


In [ ]:
from src.data_preprocessing import load_and_engineer_features

# Run domain feature engineering
df_processed = load_and_engineer_features(dataset_path)

# Verify key engineered indicators
df_processed[['project_name', 'sector', 'state', 'cost_escalation_ratio', 'financial_vs_physical_gap_pct', 'fund_drain_anomaly_flag', 'risk_level', 'root_cause']].head(5)


---
## 🎯 4. Train/Test Stratified Split & Feature Pipelines


In [ ]:
CATEGORICAL_FEATURES = ["sector", "state"]
NUMERICAL_FEATURES = [
    "original_cost_cr", "anticipated_cost_cr", "cost_overrun_cr", "cost_overrun_pct",
    "cumulative_expenditure_cr", "physical_progress_pct", "financial_progress_pct",
    "financial_vs_physical_gap_pct", "cost_escalation_ratio", "expenditure_burn_ratio",
    "log_original_cost", "fund_drain_anomaly_flag", "planned_duration_months", "report_year"
]
ALL_FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

X = df_processed[ALL_FEATURES]
y_risk = df_processed["risk_level"]
y_delay = df_processed["time_overrun_months"]
y_cause = df_processed["root_cause"]

# Risk level encoding
risk_mapping = {"Low": 0, "Medium": 1, "High": 2}
inv_risk_mapping = {0: "Low", 1: "Medium", 2: "High"}
y_risk_encoded = y_risk.map(risk_mapping)

# Root cause encoding
unique_causes = sorted(y_cause.unique())
cause_mapping = {c: i for i, c in enumerate(unique_causes)}
inv_cause_mapping = {i: c for i, c in enumerate(unique_causes)}
y_cause_encoded = y_cause.map(cause_mapping)

# Stratified 80/20 train/test split
X_train, X_test, y_risk_train, y_risk_test, y_delay_train, y_delay_test, y_cause_train, y_cause_test = train_test_split(
    X, y_risk_encoded, y_delay, y_cause_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_risk_encoded
)

# Pipeline preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("num", StandardScaler(), NUMERICAL_FEATURES)
    ]
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Train samples: {X_train_proc.shape[0]:,} | Test samples: {X_test_proc.shape[0]:,}")
print(f"Transformed Feature Dimensions: {X_train_proc.shape[1]}")


---
## 🏆 5. Model 1: Project Delay Risk Classifier (XGBoost)
Predicts whether an infrastructure project is at `Low`, `Medium`, or `High` risk of delay.


In [ ]:
# Train XGBoost Classifier for Risk Level
risk_clf = XGBClassifier(
    n_estimators=160,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    eval_metric="mlogloss"
)
risk_clf.fit(X_train_proc, y_risk_train)

# Evaluate on test set
y_risk_pred = risk_clf.predict(X_test_proc)
acc_risk = accuracy_score(y_risk_test, y_risk_pred)
f1_risk = f1_score(y_risk_test, y_risk_pred, average="macro")

print(f"✅ Risk Classifier Test Accuracy: {acc_risk * 100:.2f}%")
print(f"✅ Risk Classifier Macro F1-Score: {f1_risk:.4f}
")
print(classification_report(y_risk_test, y_risk_pred, target_names=["Low", "Medium", "High"]))


---
## ⏱️ 6. Model 2: Time Overrun Duration Regressor (XGBoost)
Predicts the exact duration of delay in months (and converted to days).


In [ ]:
# Train XGBoost Regressor for Overrun Duration
delay_reg = XGBRegressor(
    n_estimators=160,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
delay_reg.fit(X_train_proc, y_delay_train)

# Evaluate on test set
y_delay_pred = np.clip(delay_reg.predict(X_test_proc), 0.0, None)

r2_delay = r2_score(y_delay_test, y_delay_pred)
rmse_delay = np.sqrt(mean_squared_error(y_delay_test, y_delay_pred))
mae_delay = mean_absolute_error(y_delay_test, y_delay_pred)

print(f"✅ Delay Regressor R² Score: {r2_delay:.4f} ({r2_delay * 100:.2f}% Variance Explained)")
print(f"✅ Delay Regressor RMSE: {rmse_delay:.2f} months (~{rmse_delay * 30.4:.0f} days)")
print(f"✅ Delay Regressor MAE: {mae_delay:.2f} months (~{mae_delay * 30.4:.0f} days)")


---
## 🔍 7. Model 3: Root Cause Multi-Class Diagnoser (Random Forest)
Diagnoses the primary bottleneck (Land Acquisition, Forest Clearances, Funds, Contractor, Scope Change).


In [ ]:
# Train Balanced Random Forest Classifier for Root Cause
cause_clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
cause_clf.fit(X_train_proc, y_cause_train)

# Evaluate on test set
y_cause_pred = cause_clf.predict(X_test_proc)
acc_cause = accuracy_score(y_cause_test, y_cause_pred)
f1_cause = f1_score(y_cause_test, y_cause_pred, average="macro")

print(f"✅ Root Cause Classifier Test Accuracy: {acc_cause * 100:.2f}%")
print(f"✅ Root Cause Classifier Macro F1-Score: {f1_cause:.4f}
")
print(classification_report(y_cause_test, y_cause_pred, target_names=unique_causes, zero_division=0))


---
## 📋 8. Prescriptive Decision Engine & 3-Tier Escalation Matrix
The Prescriptive Engine is a deterministic rule-based matrix that consumes the predictions and produces:
- Standard Operating Procedure (SOP) action items (PM GatiShakti, RFCTLARR 2013, PARIVESH 2.0).
- 3-Tier Administrative Escalation Level:
  - **Tier 1:** Project Implementing Agency (PIA)
  - **Tier 2:** MoSPI IPMD / Inter-Ministerial Committee
  - **Tier 3:** Apex PRAGATI (PMO Level)


In [ ]:
from src.predict_pipeline import ProjectMonitoringPredictor

# Initialize unified production inference pipeline
predictor = ProjectMonitoringPredictor()

# Test Case 1: High-Risk Greenfield Highway Project
sample_highway = {
    "project_name": "DELHI-AMRITSAR-KATRA EXPRESSWAY (GREENFIELD PKG-IV)",
    "sector": "Road Transport & Highways",
    "state": "Punjab",
    "original_cost_cr": 4500.0,
    "anticipated_cost_cr": 6800.0,
    "cumulative_expenditure_cr": 3200.0,
    "physical_progress_pct": 28.5,
    "financial_progress_pct": 71.1,
    "planned_duration_months": 48.0
}

result_1 = predictor.predict_project(sample_highway)
print(json.dumps(result_1, indent=2))


---
## 🏁 9. Conclusion & Production Architecture
All 3 trained models and feature preprocessors have been exported to `models/` directory:
- `model_risk_classifier.joblib`
- `model_delay_regressor.joblib`
- `model_root_cause_classifier.joblib`
- `preprocessor.joblib`

The frontend and backend API can now import `ProjectMonitoringPredictor` from `src.predict_pipeline` with single-line inference calls!
